# YOLOv8 Segmentation Model Experiment

This notebook trains, validates, and tests a YOLOv8 segmentation model on the RoadVis/Pothole dataset.

The goal of this notebook is to evaluate YOLOv8 performance using different hyperparameter settings, including learning rate, image size, and weight decay.

## Setup Instructions

Before running this notebook, download the dataset and place it in your home directory as:

`~/roadvis_yolo/`

The dataset folder should contain:

```text
roadvis_yolo/
├── data.yaml
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

## Imports

In [ ]:
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

from ultralytics import YOLO

## Device Setup

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Training will run on CPU.")

## Project and Dataset Paths

In [ ]:
# Project and dataset paths
project_root = Path.home() / "roadvis_yolo"

data_root = project_root
data_yaml = data_root / "data.yaml"

train_images_dir = data_root / "images" / "train"
val_images_dir = data_root / "images" / "val"
test_images_dir = data_root / "images" / "test"

train_labels_dir = data_root / "labels" / "train"
val_labels_dir = data_root / "labels" / "val"
test_labels_dir = data_root / "labels" / "test"

# Output folders
results_dir = project_root / "results"
runs_dir = project_root / "runs"
plots_dir = results_dir / "plots"

results_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Dataset YAML:", data_yaml)
print("Results folder:", results_dir)
print("Runs folder:", runs_dir)

if not data_yaml.exists():
    raise FileNotFoundError(
        f"data.yaml was not found at {data_yaml}. "
        "Please download the dataset and place it inside ~/roadvis_yolo/"
    )

## Experiment Setup

Each experiment below defines its own training settings, including image size, batch size, epochs, learning rate, and weight decay.

## Dataset Sanity Check

In [ ]:
image_exts = ["*.jpg", "*.jpeg", "*.png"]


def count_images(folder):
    total = 0

    for ext in image_exts:
        total += len(list(folder.glob(ext)))

    return total


dataset_counts = {
    "train_images": count_images(train_images_dir),
    "val_images": count_images(val_images_dir),
    "test_images": count_images(test_images_dir),
    "train_labels": len(list(train_labels_dir.glob("*.txt"))),
    "val_labels": len(list(val_labels_dir.glob("*.txt"))),
    "test_labels": len(list(test_labels_dir.glob("*.txt"))),
}

print("data.yaml exists:", data_yaml.exists())
print("Train images count:", dataset_counts["train_images"])
print("Val images count:", dataset_counts["val_images"])
print("Test images count:", dataset_counts["test_images"])
print("Train labels count:", dataset_counts["train_labels"])
print("Val labels count:", dataset_counts["val_labels"])
print("Test labels count:", dataset_counts["test_labels"])

if dataset_counts["train_images"] != dataset_counts["train_labels"]:
    print("Warning: Train image and label counts do not match.")

if dataset_counts["val_images"] != dataset_counts["val_labels"]:
    print("Warning: Validation image and label counts do not match.")

if dataset_counts["test_images"] != dataset_counts["test_labels"]:
    print("Warning: Test image and label counts do not match.")

## Image and Label Matching Check

In [ ]:
def get_image_files(images_dir):
    image_files = []

    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        image_files.extend(images_dir.glob(ext))

    return sorted(image_files)


def inspect_sample_labels(images_dir, labels_dir, n=3):
    image_files = get_image_files(images_dir)

    if len(image_files) == 0:
        print(f"No images found in: {images_dir}")
        return

    for img_path in image_files[:n]:
        label_path = labels_dir / f"{img_path.stem}.txt"

        print("Image:", img_path.name)
        print("Expected label file:", label_path.name)

        if label_path.exists():
            with open(label_path, "r") as file:
                lines = file.readlines()

            print("First label lines:")

            for line in lines[:3]:
                print(" ", line.strip())
        else:
            print("Label file missing.")

        print("-" * 40)


inspect_sample_labels(train_images_dir, train_labels_dir, n=3)

## Helper Function: Extract YOLO Metrics

In [ ]:
def extract_yolo_metrics(metrics, run_id, config, train_time_min):
    result = {
        "run_id": run_id,
        "run_type": config.get("run_type"),
        "model": config["model"],
        "epochs": config["epochs"],
        "imgsz": config["imgsz"],
        "batch": config["batch"],
        "optimizer": config["optimizer"],
        "lr0": config["lr0"],
        "weight_decay": config["weight_decay"],
        "patience": config["patience"],
        "augmentation": config.get("augmentation"),
        "notes": config.get("notes"),
        "train_time_min": train_time_min,
    }

    # Detection metrics
    if hasattr(metrics, "box") and metrics.box is not None:
        result["box_precision"] = float(metrics.box.mp)
        result["box_recall"] = float(metrics.box.mr)
        result["box_map50"] = float(metrics.box.map50)
        result["box_map50_95"] = float(metrics.box.map)

    # Segmentation metrics
    if hasattr(metrics, "seg") and metrics.seg is not None:
        result["seg_precision"] = float(metrics.seg.mp)
        result["seg_recall"] = float(metrics.seg.mr)
        result["seg_map50"] = float(metrics.seg.map50)
        result["seg_map50_95"] = float(metrics.seg.map)

    return result

## Helper Functions: Train, Validate, and Save Results

In [ ]:
def validate_yolo_run(config, train_time_min):
    best_weights = runs_dir / config["run_id"] / "weights" / "best.pt"

    print("Best weights path:", best_weights)

    if not best_weights.exists():
        raise FileNotFoundError(f"best.pt not found at: {best_weights}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    best_model = YOLO(str(best_weights))

    metrics = best_model.val(
        data=str(data_yaml),
        device=device,
        batch=1
    )

    run_result = extract_yolo_metrics(
        metrics=metrics,
        run_id=config["run_id"],
        config=config,
        train_time_min=train_time_min
    )

    return run_result, metrics


def run_yolo_experiment(config):
    print(f"Starting run: {config['run_id']}")
    start_time = time.time()

    model = YOLO(config["model"])

    model.train(
        data=str(data_yaml),
        epochs=config["epochs"],
        imgsz=config["imgsz"],
        batch=config["batch"],
        device=device,
        optimizer=config["optimizer"],
        lr0=config["lr0"],
        weight_decay=config["weight_decay"],
        patience=config["patience"],
        project=str(runs_dir),
        name=config["run_id"],
        exist_ok=True
    )

    train_time_min = (time.time() - start_time) / 60.0
    print(f"Training finished in {train_time_min:.2f} minutes")

    run_result, metrics = validate_yolo_run(config, train_time_min)

    csv_path = results_dir / "experiment_log_yolov8.csv"

    if csv_path.exists():
        df_existing = pd.read_csv(csv_path)
        df_new = pd.concat([df_existing, pd.DataFrame([run_result])], ignore_index=True)
    else:
        df_new = pd.DataFrame([run_result])

    df_new = df_new.drop_duplicates(subset=["run_id"], keep="last")
    df_new.to_csv(csv_path, index=False)

    print(f"Saved experiment log to: {csv_path}")

    return run_result, metrics


def save_single_run_result(run_result, filename):
    result_path = results_dir / filename
    pd.DataFrame([run_result]).to_csv(result_path, index=False)
    print(f"Saved result to: {result_path}")
    return result_path

## Helper Function: Plot Training History

In [ ]:
def plot_training_history(run_id, title_prefix=None):
    if title_prefix is None:
        title_prefix = run_id

    run_dir = runs_dir / run_id
    results_csv = run_dir / "results.csv"

    print("Run directory:", run_dir)
    print("results.csv exists:", results_csv.exists())

    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found at: {results_csv}")

    df_history = pd.read_csv(results_csv)
    df_history.columns = [c.strip() for c in df_history.columns]

    # Loss curves
    plt.figure(figsize=(8, 5))

    if "train/box_loss" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["train/box_loss"], marker="o", label="Train Box Loss")

    if "train/seg_loss" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["train/seg_loss"], marker="o", label="Train Seg Loss")

    if "val/box_loss" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["val/box_loss"], marker="o", label="Val Box Loss")

    if "val/seg_loss" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["val/seg_loss"], marker="o", label="Val Seg Loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} Loss Curves")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / f"{run_id}_loss_curves.png", bbox_inches="tight")
    plt.show()

    # Mask mAP curves
    plt.figure(figsize=(8, 5))

    if "metrics/mAP50(M)" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["metrics/mAP50(M)"], marker="o", label="Mask mAP50")

    if "metrics/mAP50-95(M)" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["metrics/mAP50-95(M)"], marker="o", label="Mask mAP50-95")

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{title_prefix} Mask mAP")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / f"{run_id}_mask_map_curves.png", bbox_inches="tight")
    plt.show()

    # Mask precision and recall
    plt.figure(figsize=(8, 5))

    if "metrics/precision(M)" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["metrics/precision(M)"], marker="o", label="Mask Precision")

    if "metrics/recall(M)" in df_history.columns:
        plt.plot(df_history["epoch"], df_history["metrics/recall(M)"], marker="o", label="Mask Recall")

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{title_prefix} Mask Precision and Recall")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / f"{run_id}_mask_precision_recall.png", bbox_inches="tight")
    plt.show()

## Experiment V8_E00: Sanity Run

In [ ]:
config = {
    "run_id": "V8_E00",
    "run_type": "sanity",
    "model": "yolov8n-seg.pt",
    "epochs": 3,
    "imgsz": 640,
    "batch": 2,
    "patience": 2,
    "optimizer": "AdamW",
    "lr0": 1e-3,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes": "YOLOv8 sanity run, check GPU and pipeline",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E00_sanity_result.csv")
plot_training_history("V8_E00", "YOLOv8 Sanity Run")

run_result

## Experiment V8_E01: Baseline

In [ ]:
config = {
    "run_id": "V8_E01",
    "run_type": "baseline",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 640,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 1e-3,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes": "YOLOv8 official baseline",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E01_baseline_result.csv")
plot_training_history("V8_E01", "YOLOv8 Baseline")

run_result

## Experiment V8_E02: Learning Rate 1e-4

In [ ]:
config = {
    "run_id": "V8_E02",
    "run_type": "tuning",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 640,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes": "YOLOv8 learning rate tuning: lr0=1e-4",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E02_result.csv")
plot_training_history("V8_E02", "YOLOv8 Learning Rate 1e-4")

run_result

## Experiment V8_E03: Learning Rate 3e-4

In [ ]:
config = {
    "run_id": "V8_E03",
    "run_type": "tuning",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 640,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes": "YOLOv8 learning rate tuning: lr0=3e-4",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E03_result.csv")
plot_training_history("V8_E03", "YOLOv8 Learning Rate 3e-4")

run_result

## Experiment V8_E04: Image Size 512

In [ ]:
config = {
    "run_id": "V8_E04",
    "run_type": "tuning",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 512,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes": "YOLOv8 image size tuning: imgsz=512, best lr0=3e-4",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E04_result.csv")
plot_training_history("V8_E04", "YOLOv8 Image Size 512")

run_result

## Experiment V8_E05: Image Size 800

In [ ]:
config = {
    "run_id": "V8_E05",
    "run_type": "tuning",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 800,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes": "YOLOv8 image size tuning: imgsz=800, best lr0=3e-4",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E05_result.csv")
plot_training_history("V8_E05", "YOLOv8 Image Size 800")

run_result

## Experiment V8_E06: Weight Decay 5e-4

In [ ]:
config = {
    "run_id": "V8_E06",
    "run_type": "tuning",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 512,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 3e-4,
    "weight_decay": 5e-4,
    "augmentation": "default",
    "notes": "YOLOv8 weight decay tuning: weight_decay=5e-4",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E06_result.csv")
plot_training_history("V8_E06", "YOLOv8 Weight Decay 5e-4")

run_result

## Experiment V8_E07: Weight Decay 1e-3

In [ ]:
config = {
    "run_id": "V8_E07",
    "run_type": "tuning",
    "model": "yolov8n-seg.pt",
    "epochs": 10,
    "imgsz": 512,
    "batch": 2,
    "patience": 3,
    "optimizer": "AdamW",
    "lr0": 3e-4,
    "weight_decay": 1e-3,
    "augmentation": "default",
    "notes": "YOLOv8 weight decay tuning: weight_decay=1e-3",
}

run_result, metrics = run_yolo_experiment(config)
save_single_run_result(run_result, "V8_E07_result.csv")
plot_training_history("V8_E07", "YOLOv8 Weight Decay 1e-3")

run_result

## Compare YOLOv8 Experiments

In [ ]:
log_path = results_dir / "experiment_log_yolov8.csv"

if not log_path.exists():
    raise FileNotFoundError(f"Experiment log not found at: {log_path}")

df_results = pd.read_csv(log_path)

# Keep only YOLOv8 runs from this notebook.
df_results = df_results[df_results["run_id"].astype(str).str.startswith("V8_")].copy()
df_results = df_results.sort_values("run_id")

display(df_results)

# Segmentation mAP50 by run
if "seg_map50" in df_results.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(df_results["run_id"], df_results["seg_map50"], marker="o", label="Seg mAP50")
    plt.xlabel("Run ID")
    plt.ylabel("Segmentation mAP50")
    plt.title("YOLOv8 Segmentation mAP50 by Run")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / "V8_seg_map50_by_run.png", bbox_inches="tight")
    plt.show()

# Segmentation mAP50-95 by run
if "seg_map50_95" in df_results.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(df_results["run_id"], df_results["seg_map50_95"], marker="o", label="Seg mAP50-95")
    plt.xlabel("Run ID")
    plt.ylabel("Segmentation mAP50-95")
    plt.title("YOLOv8 Segmentation mAP50-95 by Run")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / "V8_seg_map50_95_by_run.png", bbox_inches="tight")
    plt.show()

# Segmentation precision by run
if "seg_precision" in df_results.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(df_results["run_id"], df_results["seg_precision"], marker="o", label="Seg Precision")
    plt.xlabel("Run ID")
    plt.ylabel("Segmentation Precision")
    plt.title("YOLOv8 Segmentation Precision by Run")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / "V8_seg_precision_by_run.png", bbox_inches="tight")
    plt.show()

# Segmentation recall by run
if "seg_recall" in df_results.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(df_results["run_id"], df_results["seg_recall"], marker="o", label="Seg Recall")
    plt.xlabel("Run ID")
    plt.ylabel("Segmentation Recall")
    plt.title("YOLOv8 Segmentation Recall by Run")
    plt.legend()
    plt.grid(True)
    plt.savefig(plots_dir / "V8_seg_recall_by_run.png", bbox_inches="tight")
    plt.show()

## Final Test Evaluation for Automatically Selected Best YOLOv8 Model

The best YOLOv8 experiment is selected from `experiment_log_yolov8.csv` using validation segmentation mAP50-95 (`seg_map50_95`). The unseen test set is only used after this selection step.

In [ ]:
# Automatically select the best YOLOv8 model using validation segmentation mAP50-95.
# This avoids hard-coding a run such as V8_E04.

experiment_log_path = results_dir / "experiment_log_yolov8.csv"
selection_metric = "seg_map50_95"

if not experiment_log_path.exists():
    raise FileNotFoundError(
        f"Experiment log not found at: {experiment_log_path}. "
        "Run the YOLOv8 training/validation experiment cells first."
    )

df_results = pd.read_csv(experiment_log_path)

# Keep only valid YOLOv8 experiment rows from this notebook.
df_results = df_results[df_results["run_id"].astype(str).str.startswith("V8_")].copy()

if df_results.empty:
    raise ValueError("No YOLOv8 runs were found in the experiment log.")

if selection_metric not in df_results.columns:
    raise ValueError(
        f"The selection metric '{selection_metric}' was not found in the experiment log. "
        f"Available columns are: {list(df_results.columns)}"
    )

# Make sure the metric is numeric and remove rows that do not have this metric.
df_results[selection_metric] = pd.to_numeric(df_results[selection_metric], errors="coerce")
df_ranked = df_results.dropna(subset=[selection_metric]).copy()

if df_ranked.empty:
    raise ValueError(f"No YOLOv8 runs have a valid value for '{selection_metric}'.")

# Highest validation segmentation mAP50-95 is selected as the best YOLOv8 run.
df_ranked = df_ranked.sort_values(selection_metric, ascending=False).reset_index(drop=True)
best_row = df_ranked.iloc[0]

best_run_id = str(best_row["run_id"])
best_imgsz = int(best_row["imgsz"])
best_weights = runs_dir / best_run_id / "weights" / "best.pt"

print("Best YOLOv8 run selected from validation results")
print("Selection metric:", selection_metric)
print("Best run ID:", best_run_id)
print("Best validation score:", best_row[selection_metric])
print("Best image size:", best_imgsz)
print("Best weights:", best_weights)
print("Weights exist:", best_weights.exists())

# Show all runs ranked by the selection metric so the choice is transparent.
summary_columns = [
    "run_id",
    "run_type",
    "imgsz",
    "batch",
    "lr0",
    "weight_decay",
    "seg_precision",
    "seg_recall",
    "seg_map50",
    "seg_map50_95",
]
summary_columns = [col for col in summary_columns if col in df_ranked.columns]
display(df_ranked[summary_columns])

if not best_weights.exists():
    raise FileNotFoundError(f"best.pt not found at: {best_weights}")

best_model = YOLO(str(best_weights))

test_metrics = best_model.val(
    data=str(data_yaml),
    split="test",
    imgsz=best_imgsz,
    device=device,
    batch=1
)

test_result = {
    "best_run_id": best_run_id,
    "selection_metric": selection_metric,
    "best_validation_score": float(best_row[selection_metric]),
    "model": best_row.get("model", "yolov8n-seg.pt"),
    "optimizer": best_row.get("optimizer"),
    "epochs": int(best_row["epochs"]) if "epochs" in best_row and pd.notna(best_row["epochs"]) else None,
    "imgsz": best_imgsz,
    "batch": int(best_row["batch"]) if "batch" in best_row and pd.notna(best_row["batch"]) else None,
    "lr0": float(best_row["lr0"]) if "lr0" in best_row and pd.notna(best_row["lr0"]) else None,
    "weight_decay": float(best_row["weight_decay"]) if "weight_decay" in best_row and pd.notna(best_row["weight_decay"]) else None,
    "test_box_precision": float(test_metrics.box.mp) if hasattr(test_metrics, "box") and test_metrics.box is not None else None,
    "test_box_recall": float(test_metrics.box.mr) if hasattr(test_metrics, "box") and test_metrics.box is not None else None,
    "test_box_map50": float(test_metrics.box.map50) if hasattr(test_metrics, "box") and test_metrics.box is not None else None,
    "test_box_map50_95": float(test_metrics.box.map) if hasattr(test_metrics, "box") and test_metrics.box is not None else None,
    "test_seg_precision": float(test_metrics.seg.mp) if hasattr(test_metrics, "seg") and test_metrics.seg is not None else None,
    "test_seg_recall": float(test_metrics.seg.mr) if hasattr(test_metrics, "seg") and test_metrics.seg is not None else None,
    "test_seg_map50": float(test_metrics.seg.map50) if hasattr(test_metrics, "seg") and test_metrics.seg is not None else None,
    "test_seg_map50_95": float(test_metrics.seg.map) if hasattr(test_metrics, "seg") and test_metrics.seg is not None else None,
    "best_weights_path": str(best_weights),
}

test_result_df = pd.DataFrame([test_result])

test_csv_path = results_dir / f"{best_run_id}_unseen_test_results.csv"
test_result_df.to_csv(test_csv_path, index=False)

print(f"Saved YOLOv8 unseen test results to: {test_csv_path}")
test_result_df

## Generate and Save Predictions for Unseen Test Images

In [ ]:
test_predictions = best_model.predict(
    source=str(test_images_dir),
    imgsz=best_imgsz,
    conf=0.25,
    save=True,
    project=str(results_dir),
    name=f"{best_run_id}_unseen_test_predictions",
    exist_ok=True
)

prediction_output_dir = results_dir / f"{best_run_id}_unseen_test_predictions"
print("Saved YOLOv8 prediction images to:", prediction_output_dir)

## Unseen Test Results Bar Chart

In [ ]:
test_csv_path = results_dir / f"{best_run_id}_unseen_test_results.csv"

if not test_csv_path.exists():
    raise FileNotFoundError(f"Test results file not found at: {test_csv_path}")

df_test = pd.read_csv(test_csv_path)
display(df_test)

metric_names = [
    "test_box_precision",
    "test_box_recall",
    "test_box_map50",
    "test_box_map50_95",
    "test_seg_precision",
    "test_seg_recall",
    "test_seg_map50",
    "test_seg_map50_95",
]

available_metrics = [m for m in metric_names if m in df_test.columns]
metric_values = [df_test.loc[0, m] for m in available_metrics]

plt.figure(figsize=(10, 5))
plt.bar([m.replace("test_", "") for m in available_metrics], metric_values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Score")
plt.title(f"{best_run_id} Unseen Test Metrics")
plt.grid(True, axis="y")
plt.savefig(plots_dir / f"{best_run_id}_unseen_test_metrics.png", bbox_inches="tight")
plt.show()

## Optional External Dataset Evaluation

This section is optional. Only run it if you have a second external dataset downloaded.

Set `external_dataset_name` to the name/source of your external dataset so the output CSV, plot title, and results table are labeled correctly.

By default, it looks for the external dataset at:

`~/external_pothole_dataset/data.yaml`

Update `external_data_yaml` if your external dataset is stored somewhere else. The cell reuses the automatically selected best YOLOv8 model from the final test section.

In [ ]:
# Optional External Dataset Evaluation
# This cell evaluates the automatically selected best YOLOv8 model on a separate external dataset.

# Name/source of the external dataset used for generalization testing.
# Change this to the real name/source of your external dataset.
external_dataset_name = "Pothole Detection v9i YOLOv8"

# Location of the external dataset data.yaml file.
# Change this path if your external dataset is stored somewhere else.
external_data_yaml = Path.home() / "Downloads" / "Pothole Detection.v9i.yolov8" / "data.yaml"

# Clean version of the dataset name for filenames.
external_dataset_slug = (
    external_dataset_name.lower()
    .replace(" ", "_")
    .replace("/", "_")
    .replace("-", "_")
)

print("External dataset name:", external_dataset_name)
print("External dataset YAML:", external_data_yaml)

if not external_data_yaml.exists():
    print("Optional external dataset not found. Skipping external evaluation.")
else:
    # Reuse the automatically selected best YOLOv8 run from the final test section.
    best_run_id_v8 = best_run_id
    best_imgsz_v8 = best_imgsz
    best_weights_v8 = best_weights

    print("YOLOv8 best run ID:", best_run_id_v8)
    print("YOLOv8 best weights:", best_weights_v8)
    print("Weights exist:", best_weights_v8.exists())

    if not best_weights_v8.exists():
        raise FileNotFoundError(f"best.pt not found at: {best_weights_v8}")

    best_model_v8 = YOLO(str(best_weights_v8))

    external_metrics_v8 = best_model_v8.val(
        data=str(external_data_yaml),
        split="test",
        imgsz=best_imgsz_v8,
        device=device,
        batch=1
    )

    external_result_v8 = {
        "model_family": "YOLOv8",
        "external_dataset_name": external_dataset_name,
        "best_run_id": best_run_id_v8,
        "imgsz": best_imgsz_v8,
        "precision_box": float(external_metrics_v8.box.mp),
        "recall_box": float(external_metrics_v8.box.mr),
        "map50_box": float(external_metrics_v8.box.map50),
        "map50_95_box": float(external_metrics_v8.box.map),
        "precision_mask": float(external_metrics_v8.seg.mp),
        "recall_mask": float(external_metrics_v8.seg.mr),
        "map50_mask": float(external_metrics_v8.seg.map50),
        "map50_95_mask": float(external_metrics_v8.seg.map),
    }

    external_df_v8 = pd.DataFrame([external_result_v8])
    display(external_df_v8)

    external_results_csv = results_dir / f"{best_run_id_v8}_{external_dataset_slug}_results.csv"
    external_df_v8.to_csv(external_results_csv, index=False)

    print(f"Saved YOLOv8 {external_dataset_name} results to:", external_results_csv)

    # Plot external dataset metrics
    metric_names = [
        "precision_mask",
        "recall_mask",
        "map50_mask",
        "map50_95_mask"
    ]

    metric_values = [
        external_result_v8["precision_mask"],
        external_result_v8["recall_mask"],
        external_result_v8["map50_mask"],
        external_result_v8["map50_95_mask"]
    ]

    plt.figure(figsize=(8, 5))
    plt.bar(metric_names, metric_values)
    plt.ylim(0, 1)
    plt.ylabel("Score")
    plt.title(f"{best_run_id_v8} Metrics on {external_dataset_name}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    external_plot_path = results_dir / f"{best_run_id_v8}_{external_dataset_slug}_metrics.png"
    plt.savefig(external_plot_path, dpi=200)
    plt.show()

    print("Saved external dataset plot to:", external_plot_path)

## Conclusion

This notebook trains and compares multiple YOLOv8 segmentation experiments. The best YOLOv8 model is selected automatically from the validation results using segmentation mAP50-95, then that selected model is evaluated on the unseen test split and prediction images are saved for review.